In [ ]:
import re
import pdg

api = pdg.connect()

In [ ]:
# Baryons in the PDG have PDG identifiers starting with 'B'.
baryons = [p for p in api.get_particles() if p.pdgid.startswith('B')]
len(baryons)

In [ ]:
def get_mass_mev(p):
    """Return the Breit-Wigner mass in MeV, falling back to parsing the name."""
    for c in p.get_children():
        if c.data_type == 'M':
            try:
                if c.value is not None:
                    return float(c.value)
            except Exception:
                pass
    for c in p.get_children():
        if c.data_type == 'SEC' and 'MASS' in c.description.upper():
            for cc in c.get_children():
                if cc.data_type == 'M':
                    try:
                        if cc.value is not None:
                            return float(cc.value)
                    except Exception:
                        pass
    m = re.search(r'\(~?(\d+)', p.description)
    if m:
        return float(m.group(1))
    return None


def get_n_gamma_br_text(p):
    """Return a string with the BR(s) to N/p/n gamma, or '-' if none recorded.

    PDG records the photon decay either as 'X --> N gamma' (charge-averaged),
    as separate 'X --> p gamma' / 'X --> n gamma' modes, or with a helicity
    qualifier (e.g. 'p gamma, helicity=1/2'). All of these are branching
    fractions and are included; only the 'helicity amplitudes' SEC entries
    (which are amplitudes, not BRs) are excluded -- those have data_type SEC.
    """
    parts = []
    for c in p.get_children():
        if c.data_type != 'BFX':
            continue
        desc = c.description
        m = re.search(r'-->\s*([Nnp])\s+gamma\b(.*)$', desc)
        if not m:
            continue
        try:
            txt = c.value_text
        except Exception:
            txt = None
        tag = m.group(2).strip(' ,')  # e.g. "helicity=1/2"
        label = f"{m.group(1)} gamma"
        if tag:
            label += f" ({tag})"
        parts.append(f"{label}: {txt if txt else 'N/A'}")
    return '; '.join(parts) if parts else '-'

In [ ]:
rows = []
for p in baryons:
    mass = get_mass_mev(p)
    if mass is None:
        continue
    rows.append((mass, p.description, get_n_gamma_br_text(p)))

rows.sort(key=lambda r: r[0])

name_w = max(len(r[1]) for r in rows)
print(f"{'Name':<{name_w}}  {'Mass [MeV]':>10}  BR(N gamma)")
print('-' * (name_w + 14 + 40))
for mass, name, br in rows:
    print(f"{name:<{name_w}}  {mass:>10.1f}  {br}")

In [ ]:
import os
import sys
import polars as pl

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.file_locations import intermediate_files_location

# GENIE Resonance_t enum (genie/Framework/ParticleData/BaryonResonance.h)
# -> PDG resonance description used in the table above.
genie_resonances = {
    0:  ('kP33_1232', 'Delta(1232)'),
    1:  ('kS11_1535', 'N(1535)'),
    2:  ('kD13_1520', 'N(1520)'),
    3:  ('kS11_1650', 'N(1650)'),
    4:  ('kD13_1700', 'N(1700)'),
    5:  ('kD15_1675', 'N(1675)'),
    6:  ('kS31_1620', 'Delta(1620)'),
    7:  ('kD33_1700', 'Delta(1700)'),
    8:  ('kP11_1440', 'N(1440)'),
    9:  ('kP33_1600', 'Delta(1600)'),
    10: ('kP13_1720', 'N(1720)'),
    11: ('kF15_1680', 'N(1680)'),
    12: ('kP31_1910', 'Delta(1910)'),
    13: ('kP33_1920', 'Delta(1920)'),
    14: ('kF35_1905', 'Delta(1905)'),
    15: ('kF37_1950', 'Delta(1950)'),
    16: ('kP11_1710', 'N(1710)'),
    17: ('kF17_1970', 'N(1970)'),
}


def parse_max_br(text):
    """Parse the BR-text built by get_n_gamma_br_text and return the largest
    upper-bound BR across modes, as a fraction (not percent).

    Handles entries of the form '(low -- high) E-2', '<X E-2', 'X+-Y E-N',
    or a bare 'X E-N'. Returns 0.0 if nothing parsable is present.
    """
    if not text or text == '-':
        return 0.0
    best = 0.0
    for segment in text.split(';'):
        if ':' not in segment:
            continue
        _, val = segment.split(':', 1)
        val = val.strip()
        if val.upper() == 'N/A':
            continue
        em = re.search(r'E([+-]?\d+)\b', val)
        scale = 10 ** int(em.group(1)) if em else 1.0
        body = re.sub(r'E[+-]?\d+\b', '', val).strip()
        m = re.search(r'\(\s*([\d.]+)\s*--\s*([\d.]+)\s*\)', body)
        if m:
            top = float(m.group(2))
        elif (m := re.search(r'<\s*([\d.]+)', body)):
            top = float(m.group(1))
        elif (m := re.search(r'([\d.]+)\s*\+-\s*([\d.]+)', body)):
            top = float(m.group(1)) + float(m.group(2))
        elif (m := re.search(r'([\d.]+)', body)):
            top = float(m.group(1))
        else:
            continue
        best = max(best, top * scale)
    return best


all_df = pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet")

agg = (
    all_df.filter(pl.col('glee_GTruth_ResNum') >= 0)
          .group_by('glee_GTruth_ResNum')
          .agg(pl.col('wc_net_weight_open_data').sum().alias('w'))
          .collect()
)
weights = dict(zip(agg['glee_GTruth_ResNum'].to_list(), agg['w'].to_list()))
total_w = sum(weights.values())

pdg_lookup = {name: (mass, br) for mass, name, br in rows}

table = []
for resnum, (label, pdg_name) in genie_resonances.items():
    w = weights.get(resnum, 0.0)
    frac = w / total_w if total_w else 0.0
    mass, br = pdg_lookup.get(pdg_name, (None, '-'))
    if mass is None:
        m = re.search(r'_(\d+)$', label)
        if m:
            mass = float(m.group(1))
    br_max = parse_max_br(br)
    eff_frac = frac * br_max
    table.append([frac, resnum, label, pdg_name, mass, br_max, eff_frac, br])

eff_total = sum(r[6] for r in table)
for r in table:
    r.insert(7, r[6] / eff_total if eff_total else 0.0)

table.sort(key=lambda r: r[0], reverse=True)

label_w = max(len(r[2]) for r in table)
name_w = max(len(r[3]) for r in table)
print(f"{'ResNum':>6}  {'GENIE label':<{label_w}}  {'PDG name':<{name_w}}  {'Mass [MeV]':>10}  {'GENIE frac':>10}  {'BR_max':>10}  {'Eff frac':>10}  {'Norm Eff':>9}  BR(N gamma)")
print('-' * (6 + 2 + label_w + 2 + name_w + 2 + 10 + 2 + 10 + 2 + 10 + 2 + 10 + 2 + 9 + 2 + 40))
for frac, resnum, label, pdg_name, mass, br_max, eff_frac, norm_eff, br in table:
    mass_s = f"{mass:>10.1f}" if mass is not None else f"{'?':>10}"
    print(f"{resnum:>6}  {label:<{label_w}}  {pdg_name:<{name_w}}  {mass_s}  {frac:>10.2%}  {br_max:>10.2e}  {eff_frac:>10.2e}  {norm_eff:>9.2%}  {br}")

In [ ]:
# Same table, restricted to NC events (wc_truth_isCC == 0).

agg_nc = (
    all_df.filter((pl.col('wc_truth_isCC') == 0) & (pl.col('glee_GTruth_ResNum') >= 0))
          .group_by('glee_GTruth_ResNum')
          .agg(pl.col('wc_net_weight_open_data').sum().alias('w'))
          .collect()
)
weights_nc = dict(zip(agg_nc['glee_GTruth_ResNum'].to_list(), agg_nc['w'].to_list()))
total_w_nc = sum(weights_nc.values())

table_nc = []
for resnum, (label, pdg_name) in genie_resonances.items():
    w = weights_nc.get(resnum, 0.0)
    frac = w / total_w_nc if total_w_nc else 0.0
    mass, br = pdg_lookup.get(pdg_name, (None, '-'))
    if mass is None:
        m = re.search(r'_(\d+)$', label)
        if m:
            mass = float(m.group(1))
    br_max = parse_max_br(br)
    eff_frac = frac * br_max
    table_nc.append([frac, resnum, label, pdg_name, mass, br_max, eff_frac, br])

eff_total_nc = sum(r[6] for r in table_nc)
for r in table_nc:
    r.insert(7, r[6] / eff_total_nc if eff_total_nc else 0.0)

table_nc.sort(key=lambda r: r[0], reverse=True)

label_w = max(len(r[2]) for r in table_nc)
name_w = max(len(r[3]) for r in table_nc)
print("NC events only (wc_truth_isCC == 0)")
print(f"{'ResNum':>6}  {'GENIE label':<{label_w}}  {'PDG name':<{name_w}}  {'Mass [MeV]':>10}  {'GENIE frac':>10}  {'BR_max':>10}  {'Eff frac':>10}  {'Norm Eff':>9}  BR(N gamma)")
print('-' * (6 + 2 + label_w + 2 + name_w + 2 + 10 + 2 + 10 + 2 + 10 + 2 + 10 + 2 + 9 + 2 + 40))
for frac, resnum, label, pdg_name, mass, br_max, eff_frac, norm_eff, br in table_nc:
    mass_s = f"{mass:>10.1f}" if mass is not None else f"{'?':>10}"
    print(f"{resnum:>6}  {label:<{label_w}}  {pdg_name:<{name_w}}  {mass_s}  {frac:>10.2%}  {br_max:>10.2e}  {eff_frac:>10.2e}  {norm_eff:>9.2%}  {br}")

In [ ]:
# Investigate glee_GTruth_DecayMode.
#
# This is the simb::GTruth::fDecayMode field copied from LArSoft's
# nutools simb::GTruth, which in turn is set from GENIE's GTruth in
# nutools/NuReweight (GENIE2GTruth.cxx). For RES events it is intended
# to encode a resonance decay channel index, but the LArSoft converter
# only fills it for a few process types (mostly charm/diffractive); for
# standard CC/NC resonance events it is left at the default -1.
#
# Verify empirically: print the distribution and a cross-tab against
# glee_GTruth_ResNum (weighted).
print("Distinct values of glee_GTruth_DecayMode (whole sample, weighted):")
print(
    all_df.group_by('glee_GTruth_DecayMode')
          .agg(pl.len().alias('n_rows'), pl.col('wc_net_weight_open_data').sum().alias('weight'))
          .sort('glee_GTruth_DecayMode', nulls_last=True)
          .collect()
)

print("\nCross-tab: DecayMode vs ResNum, restricted to GENIE resonance events (ResNum >= 0):")
print(
    all_df.filter(pl.col('glee_GTruth_ResNum') >= 0)
          .group_by(['glee_GTruth_ResNum', 'glee_GTruth_DecayMode'])
          .agg(pl.col('wc_net_weight_open_data').sum().alias('weight'))
          .sort(['glee_GTruth_ResNum', 'glee_GTruth_DecayMode'])
          .collect()
)

# Conclusion: across this full sample the column only ever takes the
# values null, -9999, and -1. Even for events with a real resonance
# (ResNum 0..16 populated), DecayMode is always -1. So this branch is
# present in the ntuple but is not actually filled with a meaningful
# decay-channel index for our MC -- it cannot be used to identify the
# resonance decay mode.

In [ ]:
# Distinguishing the resonance charge state (e.g. Delta+ vs Delta0) using
# GTruth variables only.
#
# Final-state particles in the ntuples reflect what comes out of the nucleus
# after INTRANUKE FSI, which can swap charge (e.g. pi+ -> pi0, pi- -> pi0,
# absorption, charge exchange), so they are not a reliable handle on the
# resonance charge.
#
# However, GTruth_Num{Proton,Neutron,PiPlus,Pi0,PiMinus} are pre-FSI counts
# of the GENIE primary hadronic system -- the particles emitted directly by
# the resonance decay. Charge conservation then gives the resonance charge:
#
#     Q(resonance) = NumProton + NumPiPlus - NumPiMinus
#
# (Pi0, Neutron, and any photons are neutral; eta/K from heavier states are
# also neutral or appear in pairs that cancel.)
#
# As a sanity check this matches what charge conservation in the production
# vertex predicts:
#     NC on p -> Delta+ (or N*+);   NC on n -> Delta0 (or N*0)
#     CC nu  + p -> Delta++;        CC nu  + n -> Delta+
#     CC nub + p -> Delta0;         CC nub + n -> Delta-

delta_charge_label = {2: 'Delta++', 1: 'Delta+', 0: 'Delta0', -1: 'Delta-'}

delta = (
    all_df.filter(pl.col('glee_GTruth_ResNum') == 0)
          .with_columns(
              Q_had = (pl.col('glee_GTruth_NumProton')
                       + pl.col('glee_GTruth_NumPiPlus')
                       - pl.col('glee_GTruth_NumPiMinus')),
              is_anti = pl.col('glee_GTruth_ProbePDG') < 0,
          )
)

breakdown = (
    delta.group_by(['wc_truth_isCC', 'is_anti', 'Q_had'])
         .agg(pl.col('wc_net_weight_open_data').sum().alias('w'))
         .sort(['wc_truth_isCC', 'is_anti', 'Q_had'])
         .collect()
)
print("Delta(1232) charge state distribution (weighted):")
print(f"{'isCC':>5}  {'probe':>7}  {'charge':>9}  {'weight':>12}")
print('-' * 40)
for row in breakdown.iter_rows(named=True):
    cc = 'CC' if row['wc_truth_isCC'] else 'NC'
    probe = 'nu-bar' if row['is_anti'] else 'nu'
    label = delta_charge_label.get(row['Q_had'], f"Q={row['Q_had']}")
    print(f"{cc:>5}  {probe:>7}  {label:>9}  {row['w']:>12.1f}")

# NC-only Delta(1232) split, useful for the photon analysis context.
nc_by_charge = (
    delta.filter(pl.col('wc_truth_isCC') == 0)
         .group_by('Q_had')
         .agg(pl.col('wc_net_weight_open_data').sum().alias('w'))
         .sort('Q_had')
         .collect()
)
nc_total = nc_by_charge['w'].sum()
print(f"\nNC Delta(1232) charge breakdown (total weight {nc_total:.1f}):")
for row in nc_by_charge.iter_rows(named=True):
    label = delta_charge_label.get(row['Q_had'], f"Q={row['Q_had']}")
    print(f"  {label:>9}: {row['w']:>10.1f}  ({row['w']/nc_total:>6.1%})")

# Same trick works for the heavier N* / Delta* resonances; the charge
# definition is identical because the decay products always conserve charge.
# For an N* state Q is restricted to {0, +1} and for a Delta* state to
# {-1, 0, +1, +2}, again selected by the production-vertex charge balance.